In [1]:
# Set-up and imports
import pandas as pd
import numpy as np  
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestClassifier

In [2]:
# Load datasets
shop = pd.read_csv(r"C:\Users\Glemba\Desktop\Kenyan retail analysis\cleaned_data\shop_data.csv")
holidays = pd.read_csv(r"C:\Users\Glemba\Desktop\Kenyan retail analysis\cleaned_data\holidays_data.csv")

shop['InvoiceDate'] = pd.to_datetime(shop['InvoiceDate'])
shop['Date'] = shop['InvoiceDate'].dt.date

# Aggregate total quantity sold per day
daily_sales = shop.groupby('Date')['Quantity'].sum().reset_index()
daily_sales['Date'] = pd.to_datetime(daily_sales['Date'])

# Filling missing days with 0 to ensure continuous daily date range
full_dt_range = pd.date_range(
    start=daily_sales['Date'].min(), 
    end=daily_sales['Date'].max(), 
    freq='D'
)
full_df = pd.DataFrame({'Date': full_dt_range})
full_df = full_df.merge(daily_sales, on='Date', how='left').fillna({'Quantity': 0})

# Merge holiday features
holidays['Date'] = pd.to_datetime(holidays['Date']).dt.date
holds_sbset = holidays[['Date', 'Holiday Name', 'Type']].drop_duplicates()

full_df['Date_only'] = full_df['Date'].dt.date
full_df = full_df.merge(
    holds_sbset, 
    left_on='Date_only', 
    right_on='Date', 
    how='left', 
    suffixes=('', '_hol')
)
full_df['IsHoliday'] = full_df['Holiday Name'].notna().astype(int)
full_df = full_df.drop(columns=['Date_hol', 'Holiday Name', 'Type', 'Date_only'])

# Feature Engineering (Lags, Rolling Windows, Calendar Data)
full_df['day_of_week'] = full_df['Date'].dt.dayofweek
full_df['month'] = full_df['Date'].dt.month
full_df['is_weekend'] = (full_df['day_of_week'] >= 5).astype(int)

# Sort chronologially
full_df = full_df.sort_values('Date').reset_index(drop=True)

In [ ]:
# Lag features
full_df['lag_1'] = full_df['Quantity'].shift(1)
full_df['lag_7'] = full_df['Quantity'].shift(7)
full_df['lag_14'] = full_df['Quantity'].shift(14)
full_df['roll_mean_7'] = full_df['Quantity'].shift(1).rolling(window=7).mean()
full_df['roll_std_7'] = full_df['Quantity'].shift(1).rolling(window=7).std()


In [4]:
# Create a simple baseline(Lag-7 :Predicting demand = one week ago)
full_df['bsline_lag7'] = full_df['Quantity'].shift(7)

# Train / Validation Split
## Train: Jan 1 - Sep 30 (2023) | Validation: Oct 1 - Dec 31 (2023)
split_date = pd.to_datetime('2023-10-01')
train = full_df[full_df['Date'] < split_date].copy()
val = full_df[full_df['Date'] >= split_date].copy()

# Evaluate Baseline on val set
val_ev = val.dropna(subset=['bsline_lag7'])
rmse = np.sqrt(mean_squared_error(val_ev['Quantity'], val_ev['bsline_lag7']))
mae = mean_absolute_error(val_ev['Quantity'], val_ev['bsline_lag7'])

print("--- Baseline Evaluation (Lag-7 Model) ---")
print(f"Validation RMSE: {rmse:.2f}")
print(f"Validation MAE:  {mae:.2f}")

--- Baseline Evaluation (Lag-7 Model) ---
Validation RMSE: 1.51
Validation MAE:  1.11


In [5]:
# Drop rows with NaN from shifting/rolling
model_df = full_df.dropna().copy()

In [6]:
model_df.columns

Index(['Date', 'Quantity', 'IsHoliday', 'day_of_week', 'month', 'is_weekend',
       'lag_1', 'lag_7', 'lag_14', 'roll_mean_7', 'roll_std_7', 'bsline_lag7'],
      dtype='str')

In [7]:
# Create the model
split_date = pd.to_datetime('2023-10-01')
train = model_df[model_df['Date'] < split_date]
val = model_df[model_df['Date'] >= split_date]

features = ['day_of_week', 'month', 'is_weekend', 'IsHoliday', 'lag_1', 'lag_7', 'lag_14', 'roll_mean_7', 'roll_std_7']
target = 'Quantity'

X_train, y_train = train[features], train[target]
X_val, y_val = val[features], val[target]

# Train model (Random Forest)
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

#  Evaluate model
predicts = model.predict(X_val)
ml_rmse = np.sqrt(mean_squared_error(y_val, predicts))
ml_mae = mean_absolute_error(y_val, predicts)

print("--- Model Performance Comparison ---")
print(f"Baseline (Lag-7) -> RMSE: 1.51 | MAE: 1.11")
print(f"Random Forest    -> RMSE: {ml_rmse:.2f} | MAE: {ml_mae:.2f}")

--- Model Performance Comparison ---
Baseline (Lag-7) -> RMSE: 1.51 | MAE: 1.11
Random Forest    -> RMSE: 1.30 | MAE: 0.90
